# Phase 5 — Train the baseline: pretrained ResNet-18, 3 pattern families

Trains a cell-level classifier for **Homogeneous / Speckled / Nucleolar** with 5-fold cross-validation.

**Before running:** `Runtime → Change runtime type → T4 GPU`, and make sure these two files are in
`My Drive/ANA_Pattern3/data/`:
- `cells3_100.npz` (all images, from Phase 3)
- `manifest.csv` (labels and folds, from Phase 4)

**How to use it**
1. *Pilot:* leave the settings as they are (`ROUNDS = [0]`, both brightness modes) and run all cells. This trains two models on round 0 and prints which brightness mode does better on **validation**.
2. *Full run:* set `BRIGHTNESS_MODES` to the chosen mode and `ROUNDS = [0, 1, 2, 3, 4]`, then run all cells again. Finished rounds are skipped, so nothing is trained twice.

Everything is saved to `My Drive/ANA_Pattern3/outputs/`. If Colab disconnects, just run all cells again.

In [ ]:
# ============================== 1. SETTINGS ==============================
BRIGHTNESS_MODES = ["raw", "adjusted"]   # "raw" = pixels as stored; "adjusted" = each cell stretched (see section 4)
ROUNDS = [0]                             # pilot: [0]     full run: [0, 1, 2, 3, 4]
SPLIT = "block"                          # "block" = Phase 4 folds (use this). "random" = leaky split, diagnostic only

MODEL_NAME = "resnet18"
PRETRAINED = True                        # start from ImageNet weights
CROP = 88                                # centre region of the 100x100 canvas that the network sees
INPUT_SIZE = 176                         # the crop is enlarged 2x: pretrained networks expect larger images
JITTER = 4                               # training only: shift the crop by up to 4 pixels
BRIGHTNESS_AUG = 0.10                    # training only: brightness changed by up to +-10%

BATCH_SIZE = 128
MAX_EPOCHS = 30
PATIENCE = 7                             # stop if validation has not improved for 7 epochs
LR = 3e-4                                # learning rate (step size of each weight update)
WEIGHT_DECAY = 1e-2
SEED = 42
N_FOLDS = 5
CLASS_NAMES = ["Homogeneous", "Speckled", "Nucleolar"]
SUBSAMPLE = None                         # quick tests only: keep every N-th image

In [ ]:
# COLAB-ONLY ============================== 2. COLAB + DRIVE ==============================
# Colab's own disk is fast but wiped after the session; Drive is slow but permanent.
# So: copy the two data files to Colab's disk once, and write results to Drive.
!pip install -q timm
import shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ANA_Pattern3")
DATA_DIR = Path("/content/data"); DATA_DIR.mkdir(exist_ok=True)
for name in ["cells3_100.npz", "manifest.csv"]:
    assert (DRIVE_ROOT / "data" / name).exists(), f"upload {name} to {DRIVE_ROOT / 'data'} first"
    shutil.copy(DRIVE_ROOT / "data" / name, DATA_DIR / name)
OUT_ROOT = DRIVE_ROOT / "outputs"
print("data copied to", DATA_DIR, "| results go to", OUT_ROOT)

In [ ]:
# ============================== 3. LOAD THE DATA ==============================
import json, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import timm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (balanced_accuracy_score, cohen_kappa_score, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score)
from sklearn.model_selection import StratifiedKFold

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "NO GPU - training will be very slow")

with np.load(DATA_DIR / "cells3_100.npz") as archive:
    packed = {key: archive[key] for key in archive.files}
manifest = pd.read_csv(DATA_DIR / "manifest.csv")

# Safety checks: the image array and the manifest must describe the same rows.
assert (packed["file_index"] == manifest["file_index"].values).all(), "npz and manifest are out of step"
assert (packed["labels"] == manifest["label"].values).all(), "labels differ between npz and manifest"
assert (manifest.groupby("sample_id")["fold"].nunique() == 1).all(), "a block is split across folds"

if SUBSAMPLE:                                        # quick tests only
    keep = np.arange(0, len(manifest), SUBSAMPLE)
    packed = {key: value[keep] for key, value in packed.items()}
    manifest = manifest.iloc[keep].reset_index(drop=True)

raw_images = packed["images"]                        # (N, 100, 100) uint8
labels_np = packed["labels"].astype(np.int64)
CANVAS = raw_images.shape[1]
if SPLIT == "block":
    folds_np = manifest["fold"].values
else:                                                # DIAGNOSTIC ONLY: ignores blocks, so neighbours leak
    folds_np = np.zeros(len(manifest), dtype=int)
    for f, (_, rows) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(labels_np, labels_np)):
        folds_np[rows] = f
print(f"{len(manifest)} images | canvas {CANVAS}x{CANVAS} | split = {SPLIT}")
print(pd.crosstab(folds_np, manifest["label_name"], rownames=["fold"]))

In [ ]:
# ============================== 4. THE TWO BRIGHTNESS MODES ==============================
# "raw":      pixels exactly as stored. Dim slides stay dim, bright slides stay bright.
# "adjusted": each cell is stretched on its own, so its background becomes black and its
#             brightest spots become white. This removes the overall brightness level of the
#             slide (which differs from slide to slide and lab to lab) and keeps the PATTERN.
#             Measured on the real image area only, never on the black border.
def adjust_brightness(images, heights, widths):
    out = np.zeros_like(images)
    for i in range(len(images)):
        h, w = int(heights[i]), int(widths[i])
        top, left = (CANVAS - h) // 2, (CANVAS - w) // 2
        cell = images[i, top:top + h, left:left + w].astype(np.float32)
        low, high = np.percentile(cell, [5, 99.5])    # background level, bright-spot level
        high = max(high, low + 8)                     # very flat cells: do not blow up pure noise
        out[i, top:top + h, left:left + w] = np.clip((cell - low) / (high - low), 0, 1) * 255
    return out

image_sets = {}
for mode in BRIGHTNESS_MODES:
    t0 = time.time()
    array = raw_images if mode == "raw" else adjust_brightness(raw_images, packed["heights"], packed["widths"])
    image_sets[mode] = torch.from_numpy(array).to(device)          # whole dataset lives on the GPU
    print(f"{mode:9s} ready in {time.time() - t0:4.1f}s | mean pixel {array.mean():.1f}")
labels = torch.from_numpy(labels_np).to(device)

In [ ]:
# ============================== 5. TURNING STORED IMAGES INTO NETWORK INPUT ==============================
# One function does everything, on the GPU, for a whole batch at once:
#   - cuts the centre CROP x CROP region out of the canvas and enlarges it to INPUT_SIZE
#   - training only (augment=True): random rotation by any angle, random mirror flips,
#     a shift of a few pixels and a small brightness change. A cell has no "up", so a rotated
#     or mirrored cell is an equally valid example; this gives the network more variety.
#   - subtracts the mean and divides by the std of the training pixels (one fixed formula)
def make_batch(source, rows, mean, std, augment):
    x = source[rows].float().div(255).unsqueeze(1)                  # (B, 1, 100, 100), values 0..1
    b = x.shape[0]
    scale = CROP / CANVAS
    if augment:
        angle = torch.rand(b, device=x.device) * 2 * math.pi
        flip_x = torch.where(torch.rand(b, device=x.device) < 0.5, -1.0, 1.0)
        flip_y = torch.where(torch.rand(b, device=x.device) < 0.5, -1.0, 1.0)
        shift = (torch.rand(b, 2, device=x.device) * 2 - 1) * (JITTER / (CANVAS / 2))
    else:
        angle = torch.zeros(b, device=x.device)
        flip_x = flip_y = torch.ones(b, device=x.device)
        shift = torch.zeros(b, 2, device=x.device)
    cos, sin = torch.cos(angle) * scale, torch.sin(angle) * scale
    theta = torch.stack([torch.stack([cos * flip_x, -sin * flip_y, shift[:, 0]], dim=1),
                         torch.stack([sin * flip_x,  cos * flip_y, shift[:, 1]], dim=1)], dim=1)
    grid = F.affine_grid(theta, (b, 1, INPUT_SIZE, INPUT_SIZE), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    if augment:
        gain = 1 + (torch.rand(b, 1, 1, 1, device=x.device) * 2 - 1) * BRIGHTNESS_AUG
        x = (x * gain).clamp(0, 1)
    return (x - mean) / std


@torch.no_grad()
def predict(model, source, rows, mean, std):
    # Raw network outputs (logits) for the given rows, without augmentation.
    model.eval()
    out = []
    for start in range(0, len(rows), 512):
        x = make_batch(source, rows[start:start + 512], mean, std, augment=False)
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            out.append(model(x.contiguous(memory_format=torch.channels_last)).float())
    return torch.cat(out)


def scores(y_true, logits):
    # All the metrics we report, from true labels and raw outputs.
    prob = torch.softmax(torch.as_tensor(logits), dim=1).numpy()
    y_pred = prob.argmax(1)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(len(CLASS_NAMES)), zero_division=0)
    result = {"balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
              "macro_f1": f1.mean(), "kappa": cohen_kappa_score(y_true, y_pred),
              "auc": roc_auc_score(y_true, prob, multi_class="ovr", average="macro"),
              "confusion_matrix": confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES))).tolist()}
    for i, name in enumerate(CLASS_NAMES):
        result[f"recall_{name}"], result[f"precision_{name}"], result[f"f1_{name}"] = recall[i], precision[i], f1[i]
    return {k: (float(v) if not isinstance(v, list) else v) for k, v in result.items()}

In [ ]:
# ============================== 6. ONE ROUND OF TRAINING ==============================
def run_name(mode):
    return f"{MODEL_NAME}_{mode}" + ("" if SPLIT == "block" else "_randomsplit")


def train_round(mode, k):
    out_dir = OUT_ROOT / run_name(mode) / f"round_{k}"
    if (out_dir / "metrics.json").exists():                           # resumable: finished rounds are skipped
        print(f"[{mode} round {k}] already done - skipped")
        return json.loads((out_dir / "metrics.json").read_text())
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(SEED + k); np.random.seed(SEED + k)
    source = image_sets[mode]

    # ---- who is train / validation / test in this round
    test_fold, val_fold = k, (k + 1) % N_FOLDS
    rows = {"test": np.flatnonzero(folds_np == test_fold), "val": np.flatnonzero(folds_np == val_fold),
            "train": np.flatnonzero((folds_np != test_fold) & (folds_np != val_fold))}
    rows = {name: torch.from_numpy(r).to(device) for name, r in rows.items()}
    if SPLIT == "block":                                              # no block may be in two sets
        ids = {name: set(manifest["sample_id"].values[r.cpu().numpy()]) for name, r in rows.items()}
        assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])

    # ---- normalisation numbers come from the TRAINING images only
    c = (CANVAS - CROP) // 2
    centre = source[rows["train"]][:, c:c + CROP, c:c + CROP].float().div(255)
    mean, std = centre.mean().item(), centre.std().item()
    del centre

    # ---- model, optimiser, learning-rate schedule
    model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=len(CLASS_NAMES), in_chans=1)
    model = model.to(device).to(memory_format=torch.channels_last)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(rows["train"]) / BATCH_SIZE)
    total_steps = MAX_EPOCHS * steps_per_epoch
    def lr_factor(step):                                              # 1 warm-up epoch, then a smooth (cosine) decrease
        if step < steps_per_epoch:
            return (step + 1) / steps_per_epoch
        return 0.5 * (1 + math.cos(math.pi * (step - steps_per_epoch) / max(1, total_steps - steps_per_epoch)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))        # mixed precision: faster, same result
    y_val = labels[rows["val"]].cpu().numpy()

    log, best, best_state, waited, t_start = [], -1.0, None, 0, time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        order = rows["train"][torch.randperm(len(rows["train"]), device=device)]
        loss_sum = 0.0
        for start in range(0, len(order), BATCH_SIZE):
            batch = order[start:start + BATCH_SIZE]
            x = make_batch(source, batch, mean, std, augment=True).contiguous(memory_format=torch.channels_last)
            with torch.autocast(device_type=device, enabled=(device == "cuda")):
                loss = F.cross_entropy(model(x), labels[batch])
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update(); scheduler.step()
            loss_sum += loss.item() * len(batch)

        val_logits = predict(model, source, rows["val"], mean, std)
        val_loss = F.cross_entropy(val_logits, labels[rows["val"]]).item()
        val_bal = balanced_accuracy_score(y_val, val_logits.argmax(1).cpu().numpy())
        log.append({"epoch": epoch, "train_loss": loss_sum / len(order), "val_loss": val_loss,
                    "val_balanced_accuracy": val_bal, "lr": optimizer.param_groups[0]["lr"],
                    "minutes": (time.time() - t_start) / 60})
        improved = val_bal > best
        if improved:                                                  # keep the weights of the best epoch so far
            best, waited = val_bal, 0
            best_state = {name: w.detach().cpu().clone() for name, w in model.state_dict().items()}
        else:
            waited += 1
        print(f"[{mode} round {k}] epoch {epoch:2d}  train loss {log[-1]['train_loss']:.4f}  "
              f"val loss {val_loss:.4f}  val bal.acc {val_bal:.4f}{'  *best*' if improved else ''}")
        if waited >= PATIENCE:
            print(f"  no improvement for {PATIENCE} epochs - stopping early")
            break

    # ---- reload the best epoch, then look at the TEST fold exactly once
    model.load_state_dict(best_state)
    val_logits = predict(model, source, rows["val"], mean, std).cpu().numpy()
    test_logits = predict(model, source, rows["test"], mean, std).cpu().numpy()
    metrics = {"mode": mode, "round": k, "split": SPLIT, "model": MODEL_NAME, "pretrained": PRETRAINED,
               "crop": CROP, "input_size": INPUT_SIZE, "mean": mean, "std": std,
               "best_epoch": int(np.argmax([e["val_balanced_accuracy"] for e in log]) + 1),
               "epochs_run": len(log), "minutes": log[-1]["minutes"],
               "n_train": len(rows["train"]), "n_val": len(rows["val"]), "n_test": len(rows["test"]),
               "val": scores(y_val, val_logits),
               "test": scores(labels[rows["test"]].cpu().numpy(), test_logits)}

    # ---- save everything for this round (metrics.json is written last = "round finished" flag)
    torch.save(best_state, out_dir / "best.pt")
    pd.DataFrame(log).to_csv(out_dir / "log.csv", index=False)
    per_image = []
    for name, logits in [("val", val_logits), ("test", test_logits)]:
        r = rows[name].cpu().numpy()
        frame = pd.DataFrame(logits, columns=[f"logit_{c}" for c in CLASS_NAMES])
        frame.insert(0, "label", labels_np[r]); frame.insert(0, "split", name)
        frame.insert(0, "file_index", manifest["file_index"].values[r])
        per_image.append(frame)
    pd.concat(per_image).to_csv(out_dir / "image_logits.csv", index=False)
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print(f"[{mode} round {k}] DONE  best epoch {metrics['best_epoch']}  "
          f"val bal.acc {metrics['val']['balanced_accuracy']:.4f}  test bal.acc {metrics['test']['balanced_accuracy']:.4f}  "
          f"({metrics['minutes']:.1f} min)")
    del model, optimizer
    if device == "cuda":
        torch.cuda.empty_cache()
    return metrics

In [ ]:
# ============================== 7. LOOK AT WHAT THE NETWORK WILL SEE ==============================
# Saves a picture: top row = cells as used for validation/test, bottom row = the same cells
# after random training augmentation. Stays on Drive (it contains dataset images).
for mode in BRIGHTNESS_MODES:
    sample_dir = OUT_ROOT / run_name(mode) / "samples"; sample_dir.mkdir(parents=True, exist_ok=True)
    pick = torch.from_numpy(np.concatenate([np.flatnonzero(labels_np == c)[:: max(1, (labels_np == c).sum() // 4)][:4]
                                            for c in range(len(CLASS_NAMES))])).to(device)
    torch.manual_seed(SEED)
    plain = make_batch(image_sets[mode], pick, 0.0, 1.0, augment=False).cpu().numpy()[:, 0]
    augmented = make_batch(image_sets[mode], pick, 0.0, 1.0, augment=True).cpu().numpy()[:, 0]
    fig, axes = plt.subplots(2, len(pick), figsize=(1.5 * len(pick), 3.4), facecolor="#1a1a19")
    for j in range(len(pick)):
        for row, batch in enumerate([plain, augmented]):
            axes[row, j].imshow(batch[j], cmap="gray", vmin=0, vmax=max(float(plain[j].max()), 1e-3))
            axes[row, j].axis("off")
        axes[0, j].set_title(CLASS_NAMES[labels_np[pick[j].item()]], color="#c3c2b7", fontsize=7)
    fig.suptitle(f"{mode}: top = as tested, bottom = with training augmentation (display brightened)",
                 color="#ffffff", fontsize=9, x=0.01, ha="left")
    fig.tight_layout(); fig.savefig(sample_dir / "network_input.png", dpi=110); plt.close(fig)
print("saved network_input.png for:", BRIGHTNESS_MODES)

In [ ]:
# ============================== 8. TRAIN ==============================
results = {mode: [train_round(mode, k) for k in ROUNDS] for mode in BRIGHTNESS_MODES}

In [ ]:
# ============================== 9. RESULTS ==============================
def mean_std(values):
    return f"{np.mean(values):.3f} ± {np.std(values):.3f}" if len(values) > 1 else f"{values[0]:.3f}"

INK, INK_2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e1e0d9"
lines = [f"split = {SPLIT} | model = {MODEL_NAME} (pretrained={PRETRAINED}) | crop {CROP} -> input {INPUT_SIZE} | rounds {ROUNDS}", ""]

# ---- A. choose the brightness mode on VALIDATION only
lines.append("A. VALIDATION balanced accuracy (use THIS to choose the brightness mode)")
for mode in BRIGHTNESS_MODES:
    v = [m["val"]["balanced_accuracy"] for m in results[mode]]
    lines.append(f"   {mode:9s} {mean_std(v)}   per round: " + ", ".join(f"{x:.3f}" for x in v))
lines.append("")

for mode in BRIGHTNESS_MODES:
    run_dir = OUT_ROOT / run_name(mode)
    rounds = results[mode]
    # ---- B. per-round table
    table = pd.DataFrame([{"round": m["round"], "best_epoch": m["best_epoch"], "minutes": round(m["minutes"], 1),
                           "val_bal_acc": m["val"]["balanced_accuracy"], "test_bal_acc": m["test"]["balanced_accuracy"],
                           **{f"test_recall_{n}": m["test"][f"recall_{n}"] for n in CLASS_NAMES},
                           "test_macro_f1": m["test"]["macro_f1"], "test_kappa": m["test"]["kappa"],
                           "test_auc": m["test"]["auc"]} for m in rounds])
    table.to_csv(run_dir / "summary.csv", index=False)
    lines += [f"B. {run_name(mode)}: per round", table.round(4).to_string(index=False), "",
              f"C. {run_name(mode)}: TEST, mean ± std over {len(rounds)} round(s)"]
    for column in [c for c in table.columns if c.startswith("test_")]:
        lines.append(f"   {column:<26} {mean_std(table[column].tolist())}")

    # ---- D. confusion matrix, all test folds added together
    cm = np.sum([m["test"]["confusion_matrix"] for m in rounds], axis=0)
    percent = 100 * cm / cm.sum(axis=1, keepdims=True)
    lines += ["", f"D. {run_name(mode)}: TEST confusion matrix (rows = true class, columns = predicted)",
              pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_string(),
              pd.DataFrame(percent.round(1), index=CLASS_NAMES, columns=CLASS_NAMES).to_string(), ""]
    fig, ax = plt.subplots(figsize=(4.6, 3.9), facecolor=SURFACE)
    ax.imshow(percent, cmap="Blues", vmin=0, vmax=100)
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            ax.text(j, i, f"{percent[i, j]:.1f}%\n{cm[i, j]}", ha="center", va="center", fontsize=9,
                    color="#ffffff" if percent[i, j] > 55 else INK)
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, fontsize=9, color=INK_2)
    ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES, fontsize=9, color=INK_2)
    ax.set_xlabel("predicted", color=INK_2, fontsize=9); ax.set_ylabel("true", color=INK_2, fontsize=9)
    ax.set_title(f"Test confusion matrix ({mode}, {len(rounds)} round(s))", loc="left", color=INK, fontsize=10)
    fig.tight_layout(); fig.savefig(run_dir / "confusion_matrix.png", dpi=150); plt.close(fig)

    # ---- E. learning curves: one panel per round
    fig, axes = plt.subplots(1, len(rounds), figsize=(3.6 * len(rounds), 3.2), facecolor=SURFACE, squeeze=False)
    for ax, m in zip(axes[0], rounds):
        log = pd.read_csv(run_dir / f"round_{m['round']}" / "log.csv")
        ax.plot(log["epoch"], log["train_loss"], color="#2a78d6", linewidth=2, label="training loss")
        ax.plot(log["epoch"], log["val_loss"], color="#eb6834", linewidth=2, label="validation loss")
        ax.axvline(m["best_epoch"], color="#c3c2b7", linewidth=1)
        ax.set_title(f"round {m['round']} (best epoch {m['best_epoch']})", loc="left", color=INK, fontsize=10)
        ax.set_xlabel("epoch", color=INK_2, fontsize=9); ax.set_ylim(bottom=0)
        ax.set_facecolor(SURFACE); ax.grid(axis="y", color=GRID, linewidth=0.8); ax.tick_params(colors=INK_2, labelsize=8)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
    axes[0][0].set_ylabel("loss (lower is better)", color=INK_2, fontsize=9)
    axes[0][0].legend(frameon=False, fontsize=8, labelcolor=INK)
    fig.tight_layout(); fig.savefig(run_dir / "learning_curves.png", dpi=150); plt.close(fig)
    (run_dir / "summary.txt").write_text("\n".join(lines), encoding="utf-8")

print("\n".join(lines))
print("Saved per run in", OUT_ROOT, ": summary.txt, summary.csv, confusion_matrix.png, learning_curves.png")

## What to send back

After a run, copy the **printed text of section 9** into the chat. For the full run, also download from
`My Drive/ANA_Pattern3/outputs/<run>/` and place in the project's `outputs/<run>/` folder:

- `summary.txt`, `summary.csv`, `confusion_matrix.png`, `learning_curves.png`
- from each `round_<k>/`: `log.csv`, `metrics.json`, `image_logits.csv`

Leave `best.pt` (the trained weights) and `samples/` on Drive.